<a href="https://colab.research.google.com/github/Val-Faria/tech-challenge-ia-saude/blob/main/02_otimizacao_hiperparametros_ag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Otimização de Modelos de Diagnóstico e Interpretação com IA Generativa
## Tech Challenge - Fase 2 | IA para Devs

Este projeto dá continuidade à solução desenvolvida na fase anterior para classificação de registros clínicos relacionados à tireoide. A proposta combina Algoritmos Genéticos para otimização automática de hiperparâmetros dos modelos de Machine Learning e Large Language Models (LLMs) para interpretação dos resultados em linguagem natural.

O objetivo é melhorar o desempenho dos modelos diagnósticos por meio de técnicas evolucionárias e transformar predições, probabilidades e métricas em explicações mais acessíveis e úteis para profissionais da saúde.

A solução possui caráter acadêmico e experimental. Os resultados gerados devem ser utilizados apenas como apoio à análise clínica, não substituindo a avaliação e o julgamento de profissionais da saúde.

## 1. Preparação do Ambiente

A execução do projeto foi planejada para o Google Colab, permitindo o carregamento automático dos dados e a reprodução dos experimentos em ambiente padronizado. A célula a seguir realiza as configurações iniciais do ambiente e pode incluir a instalação de dependências necessárias para a execução das etapas de otimização por Algoritmos Genéticos e interpretação dos resultados com LLMs.


In [ ]:
# Opcional - execute apenas se necessário
# !pip install openai
# !pip install transformers
# !pip install accelerate

## 2. Importação das Bibliotecas

O projeto utiliza bibliotecas para análise de dados, visualização, treinamento e avaliação de modelos de Machine Learning, além de recursos para implementação do Algoritmo Genético empregado na otimização de hiperparâmetros. O parâmetro `RANDOM_STATE` foi adotado para aumentar a reprodutibilidade dos resultados.


In [ ]:
import os
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET_COLUMN = "target"
POSITIVE_LABEL = "hypothyroid"
NEGATIVE_LABEL = "negative"

# Reprodutibilidade dos experimentos
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Seed global definida: {RANDOM_STATE}")


Seed global definida: 42


## 3. Fonte de Dados e Diretórios do Projeto

O dataset hypothyroid_final.csv é carregado automaticamente do GitHub no Google Colab ou da pasta dataset/ em ambiente local. Os diretórios necessários para armazenamento dos resultados são criados durante a execução.

In [ ]:
GITHUB_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "mvaraujo1977/TechCahllenge_Tireoide/"
    "Nirton_Afonso/dataset/hypothyroid_final.csv"
)

RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    PROJECT_ROOT = Path("/content/TechCahllenge_Tireoide")
    DATA_DIR = PROJECT_ROOT / "dataset"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    DATA_PATH = DATA_DIR / "hypothyroid_final.csv"

    print("Ambiente Google Colab detectado.")
    print(f"Baixando dataset do GitHub para: {DATA_PATH}")
    urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_PATH)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

    DATA_PATH = PROJECT_ROOT / "dataset" / "hypothyroid_final.csv"
    if not DATA_PATH.exists():
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        print("Dataset local não encontrado. Baixando do GitHub para a pasta dataset/ do projeto.")
        urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_PATH)

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Fonte GitHub: {GITHUB_DATA_URL}")
print(f"Dataset em uso: {DATA_PATH.resolve()}")
print(f"Diretório de modelos: {MODELS_DIR.resolve()}")
print(f"Diretório de figuras: {FIGURES_DIR.resolve()}")
print(f"Diretório de resultados: {RESULTS_DIR.resolve()}")

Ambiente Google Colab detectado.
Baixando dataset do GitHub para: /content/TechCahllenge_Tireoide/dataset/hypothyroid_final.csv
Fonte GitHub: https://raw.githubusercontent.com/mvaraujo1977/TechCahllenge_Tireoide/Nirton_Afonso/dataset/hypothyroid_final.csv
Dataset em uso: /content/TechCahllenge_Tireoide/dataset/hypothyroid_final.csv
Diretório de modelos: /content/TechCahllenge_Tireoide/models
Diretório de figuras: /content/TechCahllenge_Tireoide/reports/figures
